# The Embedding Atlas

*What shape is this corpus, as a distribution in 1024 dimensions?*

The chunks are embedded with `voyage-3.5` and searched by four retrieval strategies, but
nobody has looked at the space itself. Four questions:

1. **Is the space healthy?** Or is everything crammed into a narrow cone, where every
   similarity score means less than it appears to?
2. **How much of the Canon is repetition?** The Pali texts repeat stock formulas
   (*pericopes*) verbatim by design.
3. **Does it cluster?** Into topics we could name — or tag?
4. **Can we trust the map?** MN 10 and DN 22 are near-identical texts. If the embeddings
   are sound they must land on top of each other.

Nothing here changes retrieval. Findings become *hypotheses* for the eval-gated ladder.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import textwrap
from pathlib import Path

# Work from the project root however this notebook was launched.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root)
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from atlas.loader import load, check_drift
from atlas import geometry, structure, topics, pericopes

vectors, df = load()
check_drift()

print(f"{len(df):,} chunks | {df['sutta_uid'].nunique()} suttas | {vectors.shape[1]} dimensions")
print(df["nikaya"].value_counts().to_dict())

1,571 chunks | 186 suttas | 1024 dimensions
{'mn': 837, 'dn': 734}


## 1. Is the space healthy?

A baseline first, because raw numbers mean nothing without one. If 1024-dimensional unit
vectors were spread *isotropically* over the sphere, two random chunks would have a mean
cosine near **0**, and the mean vector would have length near **0**.

In [2]:
raw = geometry.anisotropy(vectors)

print(f"mean pairwise cosine : {raw['mean_pairwise_cosine']:.4f}   (isotropic would be ~0)")
print(f"mean vector norm     : {raw['mean_vector_norm']:.4f}   (isotropic would be ~0)")
print()
print(f"The single 'average chunk' direction accounts for "
      f"{raw['mean_vector_norm']**2:.0%} of a typical chunk vector.")

mean pairwise cosine : 0.6401   (isotropic would be ~0)
mean vector norm     : 0.8002   (isotropic would be ~0)

The single 'average chunk' direction accounts for 64% of a typical chunk vector.


That is a **narrow cone**, and it is the most consequential fact in this notebook.

Two chunks picked at random already score 0.64. Cosine similarity is bounded at 1.0, so
almost the entire useful range has been spent before any two chunks are compared — the
score that distinguishes *relevant* from *irrelevant* has only the sliver above 0.64 to
work in.

The shared component is identical in every vector, so it carries no information while
dominating every score. Subtracting the corpus mean removes it — the *All-but-the-Top*
correction (Mu & Viswanath, 2018).

In [3]:
centred = geometry.centre(vectors)
fixed = geometry.anisotropy(centred)

comparison = pd.DataFrame(
    {"raw": raw, "centred": fixed}
).T[["mean_pairwise_cosine", "mean_vector_norm"]].round(4)
display(comparison)

,mean_pairwise_cosine,mean_vector_norm
raw,0.6401,0.8002
centred,-0.0005,0.0108


### Where does the signal actually live?

Anisotropy alone does not say whether similarity is *useful*. That depends on the **gap**
between related and unrelated pairs. If chunks from one sutta score no higher than chunks
from different collections, the space carries no usable signal at all.

In [4]:
def signal(matrix, label):
    dist = geometry.cosine_distributions(matrix, df, sample=300_000)
    stats = dist.groupby("group")["cosine"].agg(["mean", "std"]).round(3)
    gap = stats.loc["within_sutta", "mean"] - stats.loc["cross_nikaya", "mean"]
    dist["space"] = label
    return dist, stats, gap

raw_dist, raw_stats, raw_gap = signal(vectors, "raw")
cen_dist, cen_stats, cen_gap = signal(centred, "centred")

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f"raw — gap {raw_gap:.3f}", f"centred — gap {cen_gap:.3f}"))
for col, dist in ((1, raw_dist), (2, cen_dist)):
    for group, colour in (("within_sutta", "#54A24B"), ("within_nikaya", "#4C78A8"),
                          ("cross_nikaya", "#E45756")):
        fig.add_trace(
            go.Histogram(x=dist.loc[dist["group"] == group, "cosine"], name=group,
                         marker_color=colour, opacity=0.6, nbinsx=80,
                         histnorm="probability density", showlegend=(col == 1)),
            row=1, col=col)
fig.update_layout(barmode="overlay", width=1050, height=400,
                  title="Similarity by relationship, before and after removing the common direction")
fig.show()

display(pd.concat({"raw": raw_stats, "centred": cen_stats}, axis=1))
print(f"signal gap: {raw_gap:.3f} raw -> {cen_gap:.3f} centred  ({cen_gap/raw_gap:.1f}x)")

raw        centred       
                mean    std    mean    std
group                                     
cross_nikaya   0.637  0.057  -0.008  0.117
within_nikaya  0.642  0.058   0.003  0.120
within_sutta   0.701  0.081   0.178  0.204

signal gap: 0.064 raw -> 0.186 centred  (2.9x)


Note what the middle row says in both spaces: **`within_nikaya` sits essentially on top of
`cross_nikaya`.** Knowing that two chunks come from the same collection tells you almost
nothing about whether they are similar. The Dīgha/Majjhima division is editorial — it
sorts discourses by *length*, not subject — and the embedding space confirms it carries no
semantic weight.

In [5]:
curve = geometry.pca_curve(vectors)

fig = go.Figure(go.Scatter(y=curve["cumulative"], mode="lines", line=dict(width=2.5)))
for frac, dims in [(0.50, curve["dims_50"]), (0.90, curve["dims_90"]), (0.95, curve["dims_95"])]:
    fig.add_hline(y=frac, line_dash="dot", line_color="lightgrey")
    fig.add_annotation(x=dims, y=frac, text=f"{dims} dims", showarrow=True, arrowhead=1)
fig.update_layout(title=f"Intrinsic dimensionality (of {vectors.shape[1]} nominal)",
                  xaxis_title="principal components", yaxis_title="cumulative variance",
                  width=880, height=400, showlegend=False)
fig.show()

print(f"50% of variance in {curve['dims_50']} dims | 90% in {curve['dims_90']} "
      f"| 95% in {curve['dims_95']} | nominal {vectors.shape[1]}")

50% of variance in 31 dims | 90% in 203 | 95% in 291 | nominal 1024


### Hubs: chunks retrieved for everything

In high dimensions a few points drift toward the centre of the cloud and land in almost
every neighbourhood. They are retrieval parasites — they surface for queries they have
nothing to do with. A healthy space has a flat k-occurrence distribution centred on *k*.

In [6]:
counts = geometry.hubness(vectors, k=10)
skew = geometry.hub_skew(counts)

fig = px.histogram(x=counts, nbins=70,
                   title=f"k-occurrence at k=10 — skew {skew:.2f} (0 is hub-free)",
                   labels={"x": "times a chunk appears in another chunk's top-10"})
fig.add_vline(x=10, line_dash="dash", line_color="grey", annotation_text="expected")
fig.update_layout(width=880, height=360, showlegend=False)
fig.show()

print(f"median {np.median(counts):.0f} | max {counts.max()} "
      f"| {(counts > 30).sum()} chunks appear in 30+ neighbourhoods\n")
for rank, row in enumerate(np.argsort(-counts)[:5], 1):
    r = df.iloc[row]
    print(f"{rank}. [{counts[row]:>3}x] {r.sutta_uid} #{r.chunk_index} — {r.doc_title}")
    print(f"    {' '.join(r.chunk_text.split())[:170]}...\n")

median 8 | max 75 | 30 chunks appear in 30+ neighbourhoods

1. [ 75x] mn94 #5 — With Ghoṭamukha
    He has a new ceremonial hall built to the east of the citadel. He shaves off his hair and beard, dresses in a rough antelope hide, and smears his body with ghee and oil. ...

2. [ 63x] mn39 #8 — The Longer Discourse at Assapura
    They’d think: ‘This lake is transparent, clear, and unclouded. And here are the clams and mussels, and pebbles and gravel, and schools of fish swimming about or staying s...

3. [ 61x] mn51 #7 — With Kandaraka
    When they have this spectrum of noble ethics, they experience a blameless happiness inside themselves. When they see a sight with their eyes, they don’t get caught up in ...

4. [ 58x] mn101 #8 — At Devadaha
    When they have this spectrum of noble ethics, they experience a blameless happiness inside themselves. When they see a sight with their eyes, they don’t get caught up in ...

5. [ 56x] mn103 #3 — Is This What You Think Of Me?
    After hearin

In [7]:
corr = geometry.length_vs_centrality(vectors, df)
words = df["word_count"].to_numpy(dtype=float)
centrality = vectors @ vectors.mean(axis=0)

slope, intercept = np.polyfit(words, centrality, 1)
line_x = np.array([words.min(), words.max()])

fig = go.Figure()
fig.add_trace(go.Scattergl(x=words, y=centrality, mode="markers",
                           marker=dict(size=4, opacity=0.35), name="chunks"))
fig.add_trace(go.Scatter(x=line_x, y=slope * line_x + intercept, mode="lines",
                         line=dict(color="crimson", width=2), name="least squares"))
fig.update_layout(title=f"Do longer chunks drift toward the centre?   r = {corr:.3f}",
                  xaxis_title="words in chunk", yaxis_title="similarity to corpus mean",
                  width=880, height=400)
fig.show()

## 2. How much of the Canon is repetition?

The Pali Canon is formulaic on purpose. Stock passages — the jhāna formula, the
sense-bases, "thus have I heard" — recur verbatim across hundreds of discourses, an
artefact of centuries of oral transmission before the texts were written down.

This is not dirt to be cleaned. It is a structural property, and it has a direct retrieval
consequence: a query matching a stock formula matches it in *every* sutta containing it.

Chunks merely *adjacent* within one sutta are excluded — that is the chunker splitting
continuous prose, not genuine repetition.

In [8]:
rows = []
for threshold in (0.85, 0.90, 0.95):
    _, stats = pericopes.families(vectors, df, threshold=threshold)
    rows.append({"threshold": threshold, "families": stats["n_families"],
                 "duplicate mass": f"{stats['duplicate_mass']:.1%}"})
display(pd.DataFrame(rows))

,threshold,families,duplicate mass
0,0.85,144,45.1%
1,0.90,112,19.0%
2,0.95,56,8.0%


In [9]:
family_labels, _ = pericopes.families(vectors, df, threshold=0.95)
sizes = pd.Series(family_labels).value_counts()

for family_id, size in sizes[sizes > 1].head(3).items():
    members = df.iloc[np.flatnonzero(family_labels == family_id)]
    suttas = ", ".join(sorted(members["sutta_uid"].unique())[:14])
    print(f"=== {size} chunks across: {suttas} ===")
    print(textwrap.fill(" ".join(members.iloc[0]["chunk_text"].split())[:340], 96), "...\n")

=== 5 chunks across: dn22, mn10, mn129, mn149, mn25 ===
Satisfied, the mendicants approved what the Buddha said. ...

=== 4 chunks across: dn12, mn100, mn54, mn74 ===
He said to the Buddha: “Excellent, worthy Gotama! Excellent! As if he were righting the
overturned, or revealing the hidden, or pointing out the path to the lost, or lighting a lamp in
the dark so people with clear eyes can see what’s there, worthy Gotama has made the teaching
clear in many ways. I go for refuge to the worthy Gotama, to t ...

=== 4 chunks across: mn100, mn36, mn85 ===
Because it’s a green, sappy log, and it’s lying in the water. That person will eventually get
weary and frustrated.” “In the same way, there are ascetics and brahmins who don’t live
withdrawn in body and mind from sensual pleasures. They haven’t internally given up or stilled
desire, affection, infatuation, thirst, and passion for sensual ...



### The correctness check: MN 10 against DN 22

**MN 10** (*Satipaṭṭhāna Sutta*) and **DN 22** (*Mahāsatipaṭṭhāna Sutta*) are the same
discourse — DN 22 is MN 10 with the Four Noble Truths section expanded.

That gives us free ground truth. If the embeddings are sound, every MN 10 chunk must find
its DN 22 twin at high cosine and the alignment must be monotonic. If this comes out
scattered, nothing else in this notebook can be believed.

In [10]:
matches = pericopes.align(vectors, df, "mn10", "dn22")

fig = px.scatter(matches, x="mn10_index", y="dn22_index", color="cosine",
                 color_continuous_scale="Viridis", range_color=[0.9, 1.0],
                 title="Each MN 10 chunk aligned to its best match in DN 22",
                 labels={"mn10_index": "MN 10 chunk", "dn22_index": "matched DN 22 chunk"})
fig.add_trace(go.Scatter(x=[0, 14], y=[0, 14], mode="lines", name="identity",
                         line=dict(dash="dot", color="grey")))
fig.update_traces(marker_size=12, selector=dict(mode="markers"))
fig.update_layout(width=780, height=560)
fig.show()

print(f"median cosine {matches['cosine'].median():.3f} | min {matches['cosine'].min():.3f} "
      f"| above 0.8: {(matches['cosine'] > 0.8).mean():.0%}")
print(f"spearman rank correlation: "
      f"{matches['mn10_index'].corr(matches['dn22_index'], method='spearman'):.3f}")

median cosine 1.000 | min 0.931 | above 0.8: 100%
spearman rank correlation: 1.000


The alignment tracks the identity line exactly up to MN 10 chunk 12, then steps **+8** —
and DN 22 has exactly 8 more chunks than MN 10. The expanded Four Noble Truths section
appears in the geometry as a precise insertion. The embeddings are trustworthy.

## 3. Does it cluster?

Two rules govern what follows.

**UMAP is for looking, not deciding.** It distorts density by construction, so apparent
gaps in the 2D picture can be pure artefact. Clustering therefore runs on the full
1024 dimensions, never on the projection.

**HDBSCAN is chosen because it can say "no".** k-means always returns *k* clusters — hand
it a shapeless cloud and it will invent partitions and name them. HDBSCAN labels points it
cannot assign as noise, which makes "there is no cluster structure here" an available
answer rather than an invisible one.

In [11]:
layouts = {n: structure.project(centred, n_neighbors=n) for n in (5, 15, 30)}

fig = make_subplots(rows=1, cols=3, subplot_titles=[f"n_neighbors = {n}" for n in layouts])
for col, (n, xy) in enumerate(layouts.items(), start=1):
    for nikaya, colour in (("mn", "#4C78A8"), ("dn", "#F58518")):
        mask = (df["nikaya"] == nikaya).to_numpy()
        fig.add_trace(go.Scattergl(x=xy[mask, 0], y=xy[mask, 1], mode="markers",
                                   name=nikaya.upper(), showlegend=(col == 1),
                                   marker=dict(size=3.5, color=colour, opacity=0.65)),
                      row=1, col=col)
fig.update_layout(title="The same corpus under three projection settings",
                  width=1100, height=390)
fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)
fig.show()

/Users/chetna/.pyenv/versions/3.12.4/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/chetna/.pyenv/versions/3.12.4/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/chetna/.pyenv/versions/3.12.4/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [12]:
sweep = []
for size in (10, 15, 25):
    for label, matrix in (("raw", vectors), ("centred", centred)):
        found = structure.cluster(matrix, min_cluster_size=size)
        sweep.append({"space": label, "min_cluster_size": size,
                      "clusters": len(set(found.tolist()) - {-1}),
                      "noise": f"{(found == -1).mean():.0%}"})
display(pd.DataFrame(sweep).pivot(index="min_cluster_size", columns="space",
                                  values=["clusters", "noise"]))

clusters       noise      
space             centred raw centred   raw
min_cluster_size                           
10                      5   5     76%   85%
15                      3   2     84%   91%
25                      2   0     88%  100%

### The answer is no — and that is the finding

Between 76% and 100% of chunks are unassignable. Removing the common direction helps
(85% → 76% noise) but does not change the verdict: **this corpus has no discrete topic
structure.**

That is not a tuning failure. It is what the Canon *is*. Every discourse recombines a
shared vocabulary of doctrinal formulas in overlapping proportions, so the corpus forms a
**continuum** rather than islands. There is no boundary at which "the mindfulness suttas"
end and "the ethics suttas" begin, because the same passages appear in both.

The consequence for the original motivation is direct: **cluster-derived tags would be
arbitrary.** Assigning 76% of chunks to "noise" and hard-labelling the rest would invent a
taxonomy the text does not have. The honest structures here are the *pericope families*
of §2 — which are real, discrete, and textually grounded.

In [13]:
MIN_CLUSTER_SIZE = 10          # most permissive setting; still 76% noise
cluster_labels = structure.cluster(centred, min_cluster_size=MIN_CLUSTER_SIZE)
ids, centres = structure.centroids(centred, cluster_labels)
exemplar_rows = structure.exemplars(centred, cluster_labels)

xy = layouts[15]
hover = [f"<b>{r.sutta_uid} #{r.chunk_index}</b> — {r.doc_title}<br>"
         + "<br>".join(textwrap.wrap(" ".join(r.chunk_text.split())[:400], 62))
         for r in df.itertuples()]

plot = pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "hover": hover,
                     "cluster": [str(c) if c != -1 else "unassigned" for c in cluster_labels]})
fig = px.scatter(plot, x="x", y="y", color="cluster", hover_name="hover",
                 category_orders={"cluster": [str(i) for i in ids] + ["unassigned"]},
                 color_discrete_map={"unassigned": "#D3D3D3"},
                 title=f"The map — {len(ids)} clusters over {(cluster_labels==-1).mean():.0%} unassigned space")
fig.update_traces(marker=dict(size=4.5, opacity=0.8), hovertemplate="%{hovertext}<extra></extra>")
fig.add_trace(go.Scatter(x=xy[list(exemplar_rows.values()), 0],
                         y=xy[list(exemplar_rows.values()), 1],
                         mode="markers", marker=dict(size=13, color="black", symbol="x"),
                         name="centroid exemplar"))
fig.update_layout(width=1000, height=640)
fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)
fig.show()

## 4. What little structure there is

First **without** a domain stoplist. This pass is not an oversight — the formulaic
scaffolding dominating every cluster is the §2 finding seen from another angle.

In [14]:
raw_terms = topics.ctfidf_terms(df["chunk_text"].tolist(), cluster_labels, top_n=8)
for cluster_id, words in raw_terms.items():
    print(f"cluster {cluster_id:>2}: {', '.join(words)}")

cluster  0: approved, buddha, said, satisfied, mendicants, teaching, mahākaccāna, impressive
cluster  1: mind, self, feeling, perception, consciousness, body, like, mendicant
cluster  2: worthy, buddha, gotama, refuge, follower, mendicant, day, teaching
cluster  3: gotama, brahmin, brahmins, ascetic, appropriate, good, worthy, ethical
cluster  4: wheel, king, gods, monarch, treasure, turning, buddha, obtains


In [15]:
terms = topics.ctfidf_terms(df["chunk_text"].tolist(), cluster_labels,
                            top_n=15, stopwords=topics.CANON_STOPWORDS)
passages = {c: df.iloc[row]["chunk_text"] for c, row in exemplar_rows.items()}
members = {c: df.iloc[np.flatnonzero(cluster_labels == c)]["uuid"].tolist() for c in terms}

# Groq has retired the llama-3.x chat models, so pin one from the OSS tier that
# still exists. The notebook is the composition root; the module stays model-agnostic.
from retrieval.llm_client import LLMClient

named = topics.label_clusters(terms, passages, members,
                              client=LLMClient(model_id="groq/openai/gpt-oss-120b"))

for cluster_id, label in named.items():
    size = int((cluster_labels == cluster_id).sum())
    print(f"cluster {cluster_id:>2} ({size:>4} chunks, {size/len(df):.1%}) — {label['name']}")
    print(f"     {label['gloss']}")
    print(f"     {', '.join(terms[cluster_id][:8])}\n")

cluster  0 (  13 chunks, 0.8%) — Mendicants' Satisfied Approval
     The monks express contentment and endorse the Buddha's teaching.
     approved, satisfied, teaching, mahākaccāna, shrines, impressive, exposition, answer

cluster  1 ( 296 chunks, 18.8%) — Five Aggregates Meditation
     A contemplative practice focusing on the impermanent nature of form, feeling, perception, volitional formations, and consciousness.
     mind, self, feeling, perception, consciousness, body, like, meditate

cluster  2 (  21 chunks, 1.3%) — Lay Refuge Declaration
     A lay devotee pledges lifelong refuge in the Buddha, his teaching, and the monastic community, praising the Buddha's clarity and guidance.
     worthy, gotama, refuge, follower, day, teaching, lay, clear

cluster  3 (  16 chunks, 1.0%) — Brahmin Counsel to Kūṭadanta
     Brahmins advise the affluent priest Kūṭadanta to refrain from visiting the ascetic Gotama, stressing propriety, reputation, and their own ritual expertise.
     gotama, b

## What this says

### Findings

1. **The space is a narrow cone.** 64% of every chunk vector is one shared direction, so
   two random chunks already score 0.64 and the discriminating signal lives in the thin
   band above it.
2. **Removing that direction triples the signal** — the within-sutta vs cross-nikāya gap
   goes 0.064 → 0.186 — at zero cost, since it is a single subtraction.
3. **The nikāya division is semantically empty.** `within_nikaya` and `cross_nikaya` are
   indistinguishable. Dīgha and Majjhima sort discourses by length, not subject, and the
   geometry agrees.
4. **The corpus is a continuum, not a set of topics.** 76–100% of chunks resist
   clustering in every configuration tried, in both the raw and centred spaces.
5. **Repetition is structural and large** — 19% of chunks sit in a near-duplicate family
   at 0.90, 45% at 0.85, across 112 families.
6. **Hubs exist**: k-occurrence skew 2.12, with the worst chunk appearing in 75
   neighbourhoods against a median of 8.
7. **The embeddings are trustworthy** — MN 10 aligns to DN 22 at median cosine 1.000 with
   perfect rank correlation, reproducing the known +8-chunk textual insertion.

### On the original motivation: tags

The idea that prompted this was auto-tagging chunks from cluster labels. The data says
**no** — hard clusters would impose a taxonomy the corpus does not have. The structures
that *are* real and discrete are the pericope families, which are textually grounded and
could be labelled without inventing anything.

### Hypotheses for the eval gate

Marked as hypotheses on purpose. Per the design, nothing here changes a retrieval default
until it beats the incumbent on Recall@5 and MRR in `evals/`.

- **H1 — centre the query and chunk vectors before scoring.** Biggest effect for the least
  work, and it explains why hybrid RRF already beats pure dense: FTS supplies the
  discrimination that the compressed cosine band cannot.
- **H2 — deduplicate pericope families at retrieval time.** With 19% near-duplicate mass,
  a top-5 can be five copies of one stock formula from five suttas. Collapsing a family to
  its best member should raise effective recall without touching the ranker.
- **H3 — down-weight hub chunks.** A handful of chunks appear in 30+ neighbourhoods and
  are unlikely to be what any specific query wanted.
- **H4 — drop nikāya as a retrieval facet.** Finding 3 says it carries no signal, so
  filtering by it would cost recall and buy nothing.